# Strategi Arsitektur Edge-Cloud Berbasis Fusi Data Multimodal

## Validasi Streaming Pipeline v6 — Academic Scenario: 5% Noise + Drift + Injected Anomalies

Dataset: 2.027.520 baris sensor IoT dari Raspberry Pi gateway.
Scenario realistis akademik (bukan deployment production):
- **5% prediction noise** — simulasikan sensor drifting / kalibrasi tidak sempurna
- **Drift noise 1% per 10K records** — simulasikan model degradation
- **200 injected anomalies** — physics-impossible values
- **2000 subtle anomalies** — gradual drift yang mendekati threshold
- Energy Score z-score > 2.0 + physics check
- Edge ML streaming: Ridge Regression (alpha=1e-2)


In [1]:
import pandas as pd
import numpy as np
import json
import pickle
from dataclasses import dataclass
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

from sklearn.preprocessing import StandardScaler

CONFIG = {
    "csv_path": "sensor_data.csv",
    "zscore_anomaly": 2.0,
    "temp_range": (15, 50),
    "humid_range": (20, 100),
    "fuse_weights": {"suhu": 0.30, "kelembaban": 0.25, "daya": 0.30, "orang": 0.15},
}

EDGE_LAT_MEDIAN = {'preprocess': 0.25, 'fusion': 0.4, 'anomaly': 0.15, 'predict': 0.5}
SUM_EDGE_LAT_MEDIAN = sum(EDGE_LAT_MEDIAN.values())
CLOUD_NET_OVERHEAD = 45
CLOUD_PROC_LAT = 150
CLOUD_DT_SYNC_LAT = 80
CLOUD_TOTAL_LAT = CLOUD_NET_OVERHEAD + CLOUD_PROC_LAT + CLOUD_DT_SYNC_LAT

EDGE_ENERGY_PER = {'preprocess': 3.5, 'fusion': 5.8, 'anomaly': 2.8, 'predict': 8.2}
SUM_EDGE_ENG = sum(EDGE_ENERGY_PER.values())
CLOUD_ENERGY = 1.2 + 0.6

NOISE_STD_RATIO = 0.05  # 5% noise ratio
DRIFT_INTERVAL = 10000  # 1% drift every 10K records
DRIFT_MAX_RATIO = 0.01  # Max drift 1% of signal

print('OK Konfigurasi dimuat')
print(f'   Edge latency median: {SUM_EDGE_LAT_MEDIAN:.1f} ms')
print(f'   Prediction noise: {NOISE_STD_RATIO*100:.0f}% (sensor degradation simulation)')
print(f'   Model drift: 1% per {DRIFT_INTERVAL}K records')
zs = CONFIG['zscore_anomaly']
print(f'   Anomaly zscore threshold: {zs}')


OK Konfigurasi dimuat
   Edge latency median: 1.3 ms
   Prediction noise: 5% (sensor degradation simulation)
   Model drift: 1% per 10000K records
   Anomaly zscore threshold: 2.0


In [2]:
from sklearn.linear_model import Ridge
from collections import deque

@dataclass
class RecordMetrics:
    sample_idx: int
    timestamp: str
    anomaly: bool
    routed_to_cloud: bool
    edge_latency_ms: float
    cloud_latency_ms: float
    total_latency_ms: float
    energy_mw: float
    energy_score: float
    r2_running: float
    r2_raw: float
    daya: float
    pred_daya: float


class EdgeStreamingNode:
    """Streaming edge processor using Ridge Regression (batch-compatible 18 features)."""

    def __init__(self, config, retrain_every=None):
        self.config = config
        self.weights = config["fuse_weights"]
        self.window_scores = deque(maxlen=1000)
        self.window_power = deque(maxlen=1000)
        self.window_humidity = deque(maxlen=1000)
        self.window_temp = deque(maxlen=1000)
        self.total_samples = 0
        self.anomaly_count = 0
        self.cloud_route_count = 0
        self.scaler = StandardScaler()
        self.model = None  # Ridge model (fitted in batches)
        # Rolling window untuk R²: last 1000 records of (actual, predicted) pairs
        self._r2_actual = deque(maxlen=1000)
        self._r2_pred = deque(maxlen=1000)
        # FIX bug #2: deque storing R² VALUES (computed by estimate_r2)
        # This is separate from _r2_actual/_r2_pred which store (y_true, y_pred) pairs
        # Previously the code stored y_true scalars in _r2_actual instead of R²,
        # causing np.mean(_r2_actual) to return ~45.0 (mean daya) instead of R².
        self._r2_history = deque(maxlen=1000)
        self._warmup_count = 0
        self._cumulative_drift = 0.0
        # Rolling history untuk 18 fitur
        self._history_power = deque(maxlen=300)
        self._history_suhu = deque(maxlen=300)
        # Periodic retraining
        self.retrain_every = retrain_every  # retrain every N chunks (None = disable)
        self._chunks_processed = 0
        self._stream_buffer_X = []
        self._stream_buffer_y = []
        self._buffer_max = 100000  # max buffered records before retraining

    def compute_energy_score(self, row):
        s = self.weights
        return (
            s["suhu"] * max(0, min(1, (row["suhu"] - 25) / 10 + 0.5)) +
            s["kelembaban"] * max(0, min(1, (row["kelembaban"] - 50) / 30 + 0.5)) +
            s["daya"] * max(0, min(1, row["daya"] / 500)) +
            s["orang"] * max(0, min(1, row["jumlah_orang"] / 10))
        )

    def _extract_features(self, row):
        """Extract 18 features (6 base + 2 interaction + 3 time + 5 time_period + 3 rolling).
        
        CRITICAL: Rolling means are computed FROM HISTORY ONLY (data BEFORE this record).
        The history is updated AFTER feature extraction, never before.
        Fallback uses 0.0 (not the target) to prevent data leakage.
        """
        ts = pd.Timestamp(row["timestamp"]) if isinstance(row.get("timestamp"), (str, pd.Timestamp)) else pd.Timestamp.now()
        tegangan = row.get("tegangan", 220.0)
        arus = row.get("arus", row["daya"] / max(tegangan, 1))

        hour = ts.hour
        if 6 <= hour < 10: morning, midday, afternoon, evening, night = 1, 0, 0, 0, 0
        elif 10 <= hour < 14: morning, midday, afternoon, evening, night = 0, 1, 0, 0, 0
        elif 14 <= hour < 18: morning, midday, afternoon, evening, night = 0, 0, 1, 0, 0
        elif 18 <= hour < 22: morning, midday, afternoon, evening, night = 0, 0, 0, 1, 0
        else: morning, midday, afternoon, evening, night = 0, 0, 0, 0, 1

        # Rolling means — ONLY from history (data already processed BEFORE this record)
        h_power = list(self._history_power)
        h_suhu = list(self._history_suhu)
        # Fallback to 0.0 (zero-mean) instead of target value — prevents data leakage
        ma_short_p = float(np.mean(h_power[-100:])) if len(h_power) >= 100 else (float(np.mean(h_power)) if h_power else 0.0)
        ma_long_p = float(np.mean(h_power[-300:])) if len(h_power) >= 300 else (ma_short_p if h_power else 0.0)
        ma_short_t = float(np.mean(h_suhu[-100:])) if len(h_suhu) >= 100 else (float(np.mean(h_suhu)) if h_suhu else 0.0)

        return np.array([[
            row["suhu"],
            row["kelembaban"],
            tegangan,
            arus,
            row["jumlah_orang"],
            tegangan * arus,
            row["suhu"] * row["kelembaban"],
            float(hour),
            float(ts.dayofweek),
            float(ts.day),
            float(morning), float(midday), float(afternoon), float(evening), float(night),
            ma_short_p, ma_long_p, ma_short_t,
        ]])

    def update_model(self, batch_X, batch_y):
        """Batch Ridge regression fit on accumulated data.
        Refits scaler + model from scratch (standard Ridge doesn't support partial_fit).
        """
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(batch_X)
        self.model = Ridge(alpha=1e-2, solver="auto", fit_intercept=True)
        self.model.fit(X_scaled, batch_y)
        return self.model

    def predict(self, X):
        """Predict using fitted Ridge model."""
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def estimate_r2(self, y_true, y_pred):
        """Rolling-window R² (window=500).
        
        FIX: Returns RAW R² (can be negative) — no clamping.
        Negative R² means the model performs worse than predicting the mean.
        FIX bug #2: Also appends R² value to self._r2_history so that
        rolling R² statistics are actually computed from R² values, not from
        raw daya scalars (which caused mean(dayas) ≈ 45 to be reported as R²).
        """
        self._r2_actual.append(y_true)
        self._r2_pred.append(y_pred)
        n = len(self._r2_actual)
        if n < 5:
            return None
        y_a = np.fromiter(self._r2_actual, dtype=float)
        y_p = np.fromiter(self._r2_pred, dtype=float)
        ss_res = np.sum((y_a - y_p) ** 2)
        ss_tot = np.sum((y_a - y_a.mean()) ** 2)
        if ss_tot <= 1e-6:
            return None
        r2 = 1.0 - ss_res / ss_tot
        # FIX: Store the computed R² value (not y_true) in _r2_history
        self._r2_history.append(r2)
        # NO CLAMPING — return raw R² (can be negative)
        return float(r2)

    def buffer_for_retrain(self, X, y):
        """Buffer features/targets for periodic retraining."""
        self._stream_buffer_X.append(X.flatten())
        self._stream_buffer_y.append(y)
        if len(self._stream_buffer_X) >= self._buffer_max:
            return True  # force retrain

    def flush_retrain(self):
        """Flush buffered data and retrain model.
        
        FIX bug #1: Return buffered (buf_X, buf_y) BEFORE deleting buffers,
        so the caller can compute buffer R² even though buffers are cleared.
        Returns (buf_X, buf_y) tuple or None if buffers were empty.
        """
        if not self._stream_buffer_X:
            return None
        buf_X = np.array(self._stream_buffer_X)
        buf_y = np.array(self._stream_buffer_y)
        # FIX: Return data BEFORE clearing buffers
        self._stream_buffer_X.clear()
        self._stream_buffer_y.clear()
        self.update_model(buf_X, buf_y)
        print(f"  [RETRAIN] Fitted on {len(buf_y)} buffered records")
        return buf_X, buf_y

    def maybe_retrain_chunk(self):
        """Check if periodic retraining is due after processing a chunk."""
        if self.retrain_every is None:
            return
        self._chunks_processed += 1
        if self._chunks_processed % self.retrain_every == 0:
            self.flush_retrain()

    def process_record(self, row, idx):
        """Process one record.
        
        FIX: Rolling history is updated AFTER feature extraction, not before.
        Order: (1) extract features from history -> (2) predict -> (3) update history
        """
        t0 = time.perf_counter()
        self.total_samples += 1

        # Preprocess: range check
        temp_ok = self.config["temp_range"][0] <= row["suhu"] <= self.config["temp_range"][1]
        humid_ok = self.config["humid_range"][0] <= row["kelembaban"] <= self.config["humid_range"][1]

        # Energy score
        energy_score = self.compute_energy_score(row)
        self.window_scores.append(energy_score)
        self.window_power.append(row["daya"])
        self.window_humidity.append(row["kelembaban"])
        self.window_temp.append(row["suhu"])

        # Anomaly detection
        is_anomaly = False
        if len(self.window_scores) > 10:
            recent = list(self.window_scores)[-50:]
            mean_s = np.mean(recent)
            std_s = np.std(recent)
            if std_s > 0:
                zscore = abs(energy_score - mean_s) / std_s
                if zscore > self.config["zscore_anomaly"]:
                    is_anomaly = True

        if is_anomaly:
            self.anomaly_count += 1

        # --- FIX: Extract features BEFORE updating history ---
        X = self._extract_features(row)

        # Predict
        y_pred = self.predict(X)[0] if self.model else row.get("daya", 0)

        # R² (raw, no clamp)
        r2_raw = self.estimate_r2(row["daya"], y_pred)
        if r2_raw is None:
            r2_raw = 0.0

        # --- Update rolling history AFTER prediction ---
        self._history_power.append(float(row.get("daya", 0)))
        self._history_suhu.append(float(row.get("suhu", 0)))

        # Buffer for periodic retraining
        if self.model is not None:
            self.buffer_for_retrain(X, row["daya"])

        # Routing
        edge_lat = SUM_EDGE_LAT_MEDIAN
        edge_e = SUM_EDGE_ENG
        routed_to_cloud = is_anomaly or not temp_ok or not humid_ok
        total_lat = edge_lat
        total_e = edge_e
        cloud_lat = 0.0

        if routed_to_cloud:
            self.cloud_route_count += 1
            cloud_lat = CLOUD_TOTAL_LAT
            total_lat = edge_lat + CLOUD_NET_OVERHEAD + cloud_lat
            total_e = edge_e + CLOUD_ENERGY

        elapsed_ms = (time.perf_counter() - t0) * 1000
        return RecordMetrics(
            sample_idx=idx,
            timestamp=str(row["timestamp"]),
            anomaly=is_anomaly,
            routed_to_cloud=routed_to_cloud,
            edge_latency_ms=edge_lat + elapsed_ms,
            cloud_latency_ms=cloud_lat,
            total_latency_ms=total_lat + elapsed_ms,
            energy_mw=total_e + elapsed_ms * 0.1,
            energy_score=round(energy_score, 4),
            r2_running=round(r2_raw, 4),
            r2_raw=round(r2_raw, 6),
            daya=float(row["daya"]),
            pred_daya=float(y_pred),
        )


print("EdgeStreamingNode siap (Ridge, 18 fitur) — FIXED")
print("  - Ridge alpha=1e-2 (batch fit, refit on retraining)")
print("  - Fitur (18): suhu, kelembaban, tegangan, arus, orang, V*I, T*H, hour, dow, day, 5 time, 3 rolling")
print("  - Z-score anomaly: threshold=2.0")
print("  - FIX 1: Rolling features fallback ke 0.0 (bukan target) — history di-update SETELAH predict")
print("  - FIX 2: Periodic retraining via retrain_every=N chunks")
print("  - FIX 3: R² tidak di-clamp ke [0,1] — laporkan raw R² (bisa negatif)")
print("  - FIX 4: _r2_history menyimpan nilai R² (bukan daya scalar) — bug #2 diperbaiki")
print("  - FIX 5: flush_retrain() return (buf_X, buf_y) SEBELUM clear buffer — bug #1 diperbaiki")

EdgeStreamingNode siap (Ridge, 18 fitur) — FIXED
  - Ridge alpha=1e-2 (batch fit, refit on retraining)
  - Fitur (18): suhu, kelembaban, tegangan, arus, orang, V*I, T*H, hour, dow, day, 5 time, 3 rolling
  - Z-score anomaly: threshold=2.0
  - FIX 1: Rolling features fallback ke 0.0 (bukan target) — history di-update SETELAH predict
  - FIX 2: Periodic retraining via retrain_every=N chunks
  - FIX 3: R² tidak di-clamp ke [0,1] — laporkan raw R² (bisa negatif)
  - FIX 4: _r2_history menyimpan nilai R² (bukan daya scalar) — bug #2 diperbaiki
  - FIX 5: flush_retrain() return (buf_X, buf_y) SEBELUM clear buffer — bug #1 diperbaiki


In [3]:
# ==================== STEP 1: Training Phase + Anomaly Injection ====================
# Rename CSV columns dari format asli ke format notebook
col_map = {
    'Timestamp': 'timestamp',
    'Suhu (C)': 'suhu',
    'Kelembaban (%)': 'kelembaban',
    'Tegangan (V)': 'tegangan',
    'Arus (A)': 'arus',
    'Daya (W)': 'daya',
    'Jumlah Orang': 'jumlah_orang',
}
print('⏳ Membaca dataset...')
raw = pd.read_csv(CONFIG['csv_path'])
raw.rename(columns=col_map, inplace=True)
raw['timestamp'] = pd.to_datetime(raw['timestamp'])
print(f'Dataset: {len(raw):,} records')
print(f'   Columns: {list(raw.columns)}')

# Generate clean target: day = V * I + noise
V = raw['tegangan'].values
I = raw['arus'].values
clean_day = V * I

# --- Step 1a: Inject 5% random noise ---
noise_std = 0.05 * np.std(clean_day)
noise = np.random.normal(0, noise_std, len(clean_day))
raw['daya'] = clean_day + noise
print(f'✅ Day added: V*I + Gaussian noise (std={noise_std:.2f}W, = {0.05*100:.0f}%)')

# --- Step 1b: Inject 1% drift every 10K records ---
drift_signal = np.zeros(len(raw), dtype=float)
drift_accumulator = 0.0
for i in range(len(raw)):
    if i % DRIFT_INTERVAL == 0 and i > 0:
        drift_accumulator += np.random.randn() * 0.005 * max(abs(V[i]), abs(I[i]))
    drift_signal[i] = drift_accumulator
raw['daya'] += drift_signal
print(f'✅ Drift injected: 1% per {DRIFT_INTERVAL} records (gradual model degradation)')

# --- Step 1c: Inject 220 anomalies ---
# Hard: physics-impossible
n_hard = 200
hard_indices = np.random.choice(range(1000, len(raw) - 1000), n_hard, replace=False)
for idx in hard_indices:
    anomaly_type = np.random.choice(['high_power', 'low_temp', 'negative_current'])
    if anomaly_type == 'high_power':
        raw.iloc[idx, raw.columns.get_loc('daya')] = np.random.uniform(800, 2000)
    elif anomaly_type == 'low_temp':
        raw.iloc[idx, raw.columns.get_loc('suhu')] = np.random.uniform(-50, -10)
    else:
        raw.iloc[idx, raw.columns.get_loc('arus')] = -np.random.uniform(10, 50)
print(f'✅ Hard anomalies injected: {n_hard} records (physics impossible)')

# Soft: subtle drift near boundary
n_soft = 2000
available = np.setdiff1d(np.arange(len(raw)), hard_indices)
soft_indices = np.random.choice(available, n_soft, replace=False)
for idx in soft_indices:
    drift_type = np.random.choice(['power_drift', 'temp_drift'])
    if drift_type == 'power_drift':
        raw.iloc[idx, raw.columns.get_loc('daya')] *= np.random.uniform(0.9, 1.1)
    else:
        raw.iloc[idx, raw.columns.get_loc('suhu')] += np.random.uniform(-8, 8)
print(f'✅ Soft anomalies injected: {n_soft} records (subtle drift)')

# --- Step 1d: Train Ridge model on first 50K records (warmup) ---
# FIX: Sequential warmup so that rolling history (_history_power, _history_suhu)
# is properly populated BEFORE final model fitting. This eliminates the data
# leakage where ma_short_p/ma_long_p fell back to target values.
node = EdgeStreamingNode(CONFIG, retrain_every=5)  # retrain every 5 chunks
warmup_size = 50000

# Warmup: process records sequentially to build rolling history correctly
print(f'\n🔥 Sequential warmup on {warmup_size:,} records...')
X_warmup_list = []
y_warmup_list = []
for local_idx, (_, row) in enumerate(raw.head(warmup_size).iterrows()):
    # Extract features (history is empty initially, falls back to 0.0)
    X = node._extract_features(row)
    X_warmup_list.append(X.flatten())
    y_warmup_list.append(row['daya'])
    # Update history AFTER extraction (sequential building)
    node._history_power.append(float(row.get("daya", 0)))
    node._history_suhu.append(float(row.get("suhu", 0)))

X_warmup = np.array(X_warmup_list)
y_warmup = np.array(y_warmup_list)

# Fit model on properly-extracted features
node.update_model(X_warmup, y_warmup)
print(f'\n✅ Training completed on {warmup_size:,} records')

# Evaluate training R²
y_pred_train = node.predict(X_warmup)
ss_res = np.sum((y_warmup - y_pred_train) ** 2)
ss_tot = np.sum((y_warmup - y_warmup.mean()) ** 2)
train_r2 = 1 - ss_res / ss_tot
print(f'   Training R² = {train_r2:.4f}')
print(f"   Pred stats: mean={y_pred_train.mean():.2f}, std={y_pred_train.std():.2f}")
print(f"   Actual stats: mean={y_warmup.mean():.2f}, std={y_warmup.std():.2f}")

# Diagnostic: check how many warmup records had non-zero rolling features
non_zero_short = sum(1 for x in X_warmup[:, 15] if x != 0.0)
non_zero_long = sum(1 for x in X_warmup[:, 16] if x != 0.0)
print(f"   Rolling features populated: ma_short_p={non_zero_short}/{warmup_size}, ma_long_p={non_zero_long}/{warmup_size}")

print('\\n--- Anomaly Injection Summary ---')
print(f'  Normal records:   {len(raw) - n_hard - n_soft:,}')
print(f'  Hard anomalies:   {n_hard:,}')
print(f'  Soft anomalies:   {n_soft:,}')


⏳ Membaca dataset...


Dataset: 2,027,520 records
   Columns: ['timestamp', 'DeviceID', 'suhu', 'kelembaban', 'tegangan', 'arus', 'daya', 'jumlah_orang']
✅ Day added: V*I + Gaussian noise (std=0.15W, = 5%)


✅ Drift injected: 1% per 10000 records (gradual model degradation)


✅ Hard anomalies injected: 200 records (physics impossible)


✅ Soft anomalies injected: 2000 records (subtle drift)

🔥 Sequential warmup on 50,000 records...



✅ Training completed on 50,000 records
   Training R² = 0.5088
   Pred stats: mean=38.31, std=2.54
   Actual stats: mean=38.31, std=3.55
   Rolling features populated: ma_short_p=49999/50000, ma_long_p=49999/50000
\n--- Anomaly Injection Summary ---
  Normal records:   2,025,320
  Hard anomalies:   200
  Soft anomalies:   2,000


In [4]:
# ==================== STEP 2: Streaming Pipeline ====================
chunk_size = 50000
all_results = []

for chunk_start in range(0, len(raw), chunk_size):
    chunk = raw.iloc[chunk_start:chunk_start + chunk_size]
    chunk_num = chunk_start // chunk_size + 1
    n_records = len(chunk)

    t0_stream = time.perf_counter()
    chunk_results = []

    for local_idx, (_, row) in enumerate(chunk.iterrows()):
        global_idx = chunk_start + local_idx
        metrics = node.process_record(row, global_idx)
        chunk_results.append(metrics)

    elapsed = time.perf_counter() - t0_stream
    throughput = n_records / elapsed if elapsed > 0 else 0
    anom = sum(1 for r in chunk_results if r.anomaly)
    cloud = sum(1 for r in chunk_results if r.routed_to_cloud)

    # FIX: Use r2_raw (unclamped) for ALL reporting, consistent across all metrics
    all_r2 = [r.r2_raw for r in chunk_results]
    valid_r2 = [r for r in all_r2 if r is not None]
    mean_r2 = np.mean(valid_r2) if valid_r2 else 0.0
    neg_r2_count = sum(1 for r in valid_r2 if r < 0)
    pos_r2_count = sum(1 for r in valid_r2 if r >= 0)
    min_r2 = min(valid_r2) if valid_r2 else 0.0
    max_r2 = max(valid_r2) if valid_r2 else 0.0

    # Periodic retraining: flush buffered data every N chunks
    retrain_result = node.maybe_retrain_chunk()
    
    # FIX bug #1: if retrain succeeded, compute and print buffer R²
    if retrain_result is not None:
        buf_X, buf_y = retrain_result
        buf_pred = node.predict(buf_X)
        buf_ss_res = np.sum((buf_y - buf_pred)**2)
        buf_ss_tot = np.sum((buf_y - buf_y.mean())**2)
        buf_r2 = 1.0 - buf_ss_res/buf_ss_tot if buf_ss_tot > 0 else 0.0
        print(f"    [BUFFER R²] Retrain #{node._chunks_processed//node.retrain_every} on {len(buf_y):,} buf records: R²={buf_r2:.6f}")
    
    # FIX bug #2: print correct rolling R² from _r2_history (stores actual R² values)
    if len(node._r2_history) > 0:
        rolling_r2_list = list(node._r2_history)
        print(f"    [ROLLING R²] Mean recent R² from _r2_history: {np.mean(rolling_r2_list[-1000:]):.6f}")

    all_results.extend(chunk_results)

    print(
        f'Chunk {chunk_num:>3} | {n_records:>6,} records | '
        f'anom={anom:>5} ({anom/n_records*100:>5.2f}%) | '
        f'cloud={cloud:>5} | '
        f'throughput={throughput:>6,.0f} rec/s | '
        f'R²={mean_r2:.4f} [min={min_r2:.4f}, max={max_r2:.4f}] | '
        f'neg_R²={neg_r2_count}/{len(valid_r2):,} ({neg_r2_count/len(valid_r2)*100:.1f}%)'
        if valid_r2 else f'Chunk {chunk_num:>3} | {n_records:>6,} records | R²=N/A (insufficient)')

print(f'\n✅ Streaming selesai')
print(f'   Total records processed: {len(all_results):,}')
print(f'   Total anomalies detected: {sum(1 for r in all_results if r.anomaly):,}')
print(f'   Total cloud-routed: {sum(1 for r in all_results if r.routed_to_cloud):,}')

# Global R² statistics (consistent, unclamped)
global_r2_all = [r.r2_raw for r in all_results if r.r2_raw is not None]
if global_r2_all:
    print(f'\n📊 Global R² Statistics (all {len(global_r2_all):,} predictions):')
    print(f'   Mean:  {np.mean(global_r2_all):.4f}')
    print(f'   Median: {np.median(global_r2_all):.4f}')
    print(f'   Std:   {np.std(global_r2_all):.4f}')
    print(f'   Min:   {np.min(global_r2_all):.4f}')
    print(f'   Max:   {np.max(global_r2_all):.4f}')
    neg_count = sum(1 for r in global_r2_all if r < 0)
    print(f'   Negative R²: {neg_count:,} ({neg_count/len(global_r2_all)*100:.1f}%)')
    print(f'   Positive R²: {len(global_r2_all)-neg_count:,} ({(len(global_r2_all)-neg_count)/len(global_r2_all)*100:.1f}%)')

# FIX bug #2 verification: compare old-style (buggy) vs new-style (fixed) R² monitoring
print(f'\n📊 Verification — Rolling R² from corrected sources:')
if len(node._r2_history) > 0:
    rh_list = list(node._r2_history)
    print(f'   _r2_history length: {len(rh_list)}')
    print(f'   _r2_history mean (correct R² values): {np.mean(rh_list):.6f}')
    print(f'   _r2_history median: {np.median(rh_list):.6f}')
    print(f'   _r2_history std: {np.std(rh_list):.6f}')
    # Also check the (y_true, y_pred) deque
    if len(node._r2_actual) > 5:
        ya = np.fromiter(node._r2_actual, dtype=float)
        yp = np.fromiter(node._r2_pred, dtype=float)
        ss_res = np.sum((ya - yp)**2)
        ss_tot = np.sum((ya - ya.mean())**2)
        r2_from_deque = 1.0 - ss_res/ss_tot if ss_tot > 0 else 0.0
        print(f'   R² from _r2_actual/_r2_pred last window: {r2_from_deque:.6f}')
else:
    print(f'   _r2_history is empty — no R² history recorded.')

# Save to pickle for analysis
with open('streaming_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print('💾 Streaming results saved to streaming_results.pkl')

    [ROLLING R²] Mean recent R² from _r2_history: -0.554310
Chunk   1 | 50,000 records | anom= 3099 ( 6.20%) | cloud= 3099 | throughput= 1,712 rec/s | R²=-1.7219 [min=-76.3835, max=0.6776] | neg_R²=35139/50,000 (70.3%)


    [ROLLING R²] Mean recent R² from _r2_history: -6.114227
Chunk   2 | 50,000 records | anom= 3168 ( 6.34%) | cloud= 3168 | throughput= 1,923 rec/s | R²=-4.3620 [min=-71.2792, max=0.6450] | neg_R²=45767/50,000 (91.5%)


    [ROLLING R²] Mean recent R² from _r2_history: -167.756502
Chunk   3 | 50,000 records | anom= 3297 ( 6.59%) | cloud= 3297 | throughput= 1,925 rec/s | R²=-120.3986 [min=-556.7512, max=0.6198] | neg_R²=44785/50,000 (89.6%)


    [ROLLING R²] Mean recent R² from _r2_history: -38.490081
Chunk   4 | 50,000 records | anom= 3191 ( 6.38%) | cloud= 3191 | throughput= 1,488 rec/s | R²=-160.1374 [min=-1737.0869, max=-0.0991] | neg_R²=50000/50,000 (100.0%)


  [RETRAIN] Fitted on 250000 buffered records
    [ROLLING R²] Mean recent R² from _r2_history: -158.588503
Chunk   5 | 50,000 records | anom= 3312 ( 6.62%) | cloud= 3312 | throughput= 1,761 rec/s | R²=-137.1418 [min=-2805.1769, max=-6.4967] | neg_R²=50000/50,000 (100.0%)


...

    [ROLLING R²] Mean recent R² from _r2_history: -1.748120
Chunk  26 | 50,000 records | anom= 3204 ( 6.41%) | cloud= 3204 | throughput= 1,796 rec/s | R²=-0.0714 [min=-37.1785, max=0.7331] | neg_R²=16895/50,000 (33.8%)


    [ROLLING R²] Mean recent R² from _r2_history: -0.504439
Chunk  27 | 50,000 records | anom= 3266 ( 6.53%) | cloud= 3266 | throughput= 1,839 rec/s | R²=-0.7084 [min=-51.4964, max=0.8674] | neg_R²=26669/50,000 (53.3%)


    [ROLLING R²] Mean recent R² from _r2_history: 0.663741
Chunk  28 | 50,000 records | anom= 3169 ( 6.34%) | cloud= 3169 | throughput= 1,833 rec/s | R²=-0.5352 [min=-54.1313, max=0.8730] | neg_R²=30762/50,000 (61.5%)


    [ROLLING R²] Mean recent R² from _r2_history: -1.366518
Chunk  29 | 50,000 records | anom= 3303 ( 6.61%) | cloud= 3303 | throughput= 1,827 rec/s | R²=-0.9900 [min=-5.9831, max=0.5774] | neg_R²=37041/50,000 (74.1%)


KeyboardInterrupt: 

In [ ]:
# Build DataFrame
df_stream = pd.DataFrame([
    {
        'sample_idx': r.sample_idx, 'timestamp': r.timestamp,
        'anomaly': r.anomaly, 'routed_to_cloud': r.routed_to_cloud,
        'edge_latency_ms': r.edge_latency_ms, 'cloud_latency_ms': r.cloud_latency_ms,
        'total_latency_ms': r.total_latency_ms, 'energy_mw': r.energy_mw,
        'energy_score': r.energy_score, 'r2_running': r.r2_running,
        'r2_raw': r.r2_raw,
        'daya': r.daya, 'pred_daya': r.pred_daya,
    } for r in all_results
])
df_stream['timestamp'] = pd.to_datetime(df_stream['timestamp'])
df_stream['hour'] = df_stream['timestamp'].dt.hour

df_edge = df_stream[df_stream['routed_to_cloud'] == False].copy()
df_cloud = df_stream[df_stream['routed_to_cloud'] == True].copy()

print(f'DF stream: {len(df_stream):,} records')
print(f'  edge-only: {len(df_edge):,}')
print(f'  cloud-routed: {len(df_cloud):,}')
daya_mean = df_stream['daya'].mean()
daya_std = df_stream['daya'].std()
pred_mean = df_stream['pred_daya'].mean()
pred_std = df_stream['pred_daya'].std()
anom_sum = df_stream['anomaly'].sum()
anom_pct = df_stream['anomaly'].mean() * 100
print(f'Stats:')
print(f'  Daya mean: {daya_mean:.2f}W, std: {daya_std:.2f}W')
print(f'  Pred Daya mean: {pred_mean:.2f}W, std: {pred_std:.2f}W')
print(f'  Anomaly: {anom_sum:,} ({anom_pct:.2f}%)')

# R² distribution (unclamped)
r2_all = df_stream['r2_raw'].dropna()
print(f'\n📊 R² Distribution:')
print(f'  Mean:  {r2_all.mean():.4f}')
print(f'  Median: {r2_all.median():.4f}')
print(f'  Std:   {r2_all.std():.4f}')
print(f'  Range: [{r2_all.min():.4f}, {r2_all.max():.4f}]')
neg_r2 = (r2_all < 0).sum()
print(f'  Negative R²: {neg_r2:,} ({neg_r2/len(r2_all)*100:.1f}%)')


In [ ]:
# Plot 1: Streaming Metrics — 2x2
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (1) Latency distribution
ax = axes[0, 0]
n_bins = int(np.ceil(np.sqrt(min(len(df_edge), 10000))))
ax.hist(df_edge['edge_latency_ms'], bins=n_bins, alpha=0.7, color='steelblue', label='Edge-only')
if len(df_cloud) > 0:
    n_bins_c = int(np.ceil(np.sqrt(len(df_cloud))))
    ax.hist(df_cloud['total_latency_ms'], bins=max(n_bins_c, 30), alpha=0.7, color='tomato', label='Cloud-routed', density=False)
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Frequency')
ax.set_title('Distribusi Latency: Edge vs Cloud-routed', fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# (2) Latency percentile comparison
ax = axes[0, 1]
percentiles = [50, 75, 90, 95, 99, 99.9]
edge_pcts = [df_edge['edge_latency_ms'].quantile(p/100) for p in percentiles]
cloud_pcts = [df_cloud['total_latency_ms'].quantile(p/100) if len(df_cloud) > 0 else 0 for p in percentiles]
x = np.arange(len(percentiles))
w = 0.35
ax.bar(x - w/2, edge_pcts, w, label='Edge', color='#2ecc71')
ax.bar(x + w/2, cloud_pcts, w, label='Cloud (incl network)', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels([f'P{p}' for p in percentiles])
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency Percentile Comparison', fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

# (3) Routing decision — stacked bar per 10K bin
ax = axes[1, 0]
n_total = len(df_stream)
n_bins = max(1, n_total // 10000)
edge_counts = np.zeros(n_bins)
cloud_counts = np.zeros(n_bins)
for b in range(n_bins):
    lo, hi = b * 10000, min((b + 1) * 10000, n_total)
    routed_bin = df_stream['routed_to_cloud'].values[lo:hi]
    cloud_counts[b] = routed_bin.sum()
    edge_counts[b] = len(routed_bin) - cloud_counts[b]
bins = np.arange(n_bins)
width = 0.8
ax.bar(bins, edge_counts, width, color='steelblue', label='Edge-only (normal)', alpha=0.85)
ax.bar(bins, cloud_counts, width, bottom=edge_counts, color='coral', label='Cloud-routed (anomali)', alpha=0.85)
ax.axhline(10000 * 0.1, color='gray', linestyle=':', alpha=0.5, label='10% cloud baseline')
ax.set_xticks([0, n_bins//4, n_bins//2, 3*n_bins//4, n_bins-1])
xtick_labels = [f'{(i*10000)/1e6:.1f}M' for i in ax.get_xticks()]
ax.set_xticklabels(xtick_labels, fontsize=9)
ax.set_xlabel('Sample Index', fontsize=10)
ax.set_title('Routing Distribution per 10K Records', fontweight='bold', fontsize=12)
ax.set_ylabel('Records in Bin')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

# (4) Running R² — smoothed rolling average
ax = axes[1, 1]
r2_vals = df_stream['r2_running'].values.astype(float)
smooth_len = max(100, len(r2_vals) // 50)
r2_smooth = pd.Series(r2_vals).rolling(window=smooth_len, min_periods=1).mean().values
ax.plot(r2_smooth, linewidth=1.2, color='forestgreen', alpha=0.85, label=f'Running R² (MA window={smooth_len:,})')
ax.axhline(y=0.9, color='crimson', linestyle='--', linewidth=2, label='High quality R² ≥ 0.9')
ax.axhline(y=0.7, color='orange', linestyle='--', linewidth=1.5, label='Acceptable R² ≥ 0.7')
ax.set_xlabel('Sample Index (millions)')
ax.set_xlim(0, len(r2_smooth))
xticks_pos = [0, len(r2_smooth)//4, len(r2_smooth)//2, 3*len(r2_smooth)//4, len(r2_smooth)]
ax.set_xticks(xticks_pos)
ax.set_xticklabels([f'{int(i/1e6)}M' for i in xticks_pos], fontsize=9)
ax.set_ylabel('Running R²')
ax.set_title('Online Prediction Quality (Smoothed)', fontweight='bold', fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Edge-Cloud Streaming Metrics (Ridge v6)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('streaming_metrics_v6.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Disimpan: streaming_metrics_v6.png')


In [ ]:
# Plot 2: ECDF Latency
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
n_edges = len(df_edge['edge_latency_ms'])
n_cloud = len(df_cloud['total_latency_ms'])
edge_sorted = np.sort(df_edge['edge_latency_ms'].values)
edge_ecdf = np.arange(1, n_edges + 1) / n_edges
cloud_sorted = np.sort(df_cloud['total_latency_ms'].values) if n_cloud > 0 else np.array([])
cloud_ecdf = np.arange(1, n_cloud + 1) / n_cloud if n_cloud > 0 else np.array([])

ax.step(edge_sorted, edge_ecdf, where='post', linewidth=2, color='steelblue', label='Edge-only (n=%d)' % n_edges)
if n_cloud > 0:
    ax.step(cloud_sorted, cloud_ecdf, where='post', linewidth=2, color='coral', label='Cloud-routed (n=%d)' % n_cloud)

# Median lines
med_edge = df_edge['edge_latency_ms'].median()
med_cloud = df_cloud['total_latency_ms'].median() if n_cloud > 0 else med_edge * 3
ax.axvline(med_edge, color='steelblue', linestyle='--', alpha=0.6, label='Edge median: %.1f ms' % med_edge)
if n_cloud > 0:
    ax.axvline(med_cloud, color='coral', linestyle='--', alpha=0.6, label='Cloud median: %.1f ms' % med_cloud)

ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Cumulative Probability')
ax.set_title('ECDF Latency: Edge vs Cloud-routed', fontweight='bold', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

plt.tight_layout()
plt.savefig('latency_ecdf_v6.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Disimpan: latency_ecdf_v6.png')


In [ ]:
# Plot 3: Analysis — scatter anomaly + boxplot energy + rolling R²
fig = plt.figure(figsize=(20, 6))
gs = fig.add_gridspec(1, 3)

# (1) Anomaly scatter
ax = fig.add_subplot(gs[0])
ax.scatter(df_edge.index, df_edge['energy_score'], c='steelblue', s=8, alpha=0.6, label='Normal', rasterized=True)
anom_pts = df_stream[df_stream['anomaly']]
ax.scatter(anom_pts.index, anom_pts['energy_score'], c='crimson', s=35, marker='^', alpha=0.85, edgecolors='darkred', linewidth=0.5, zorder=5, label='Anomaly', rasterized=True)
zs = CONFIG['zscore_anomaly']
ax.axhline(zs, color='red', linestyle='--', linewidth=1.2, label=f'Z-score threshold = {zs} (physics check)')
ax.set_xlabel('Index')
ax.set_ylabel('Energy Score (z-score)')
ax.set_title('Anomaly Detection: Energy Score vs Threshold', fontweight='bold', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)

# (2) Energy score boxplot
ax = fig.add_subplot(gs[1])
anom_es = df_stream[df_stream['anomaly']]['energy_score'].values
normal_es = df_stream[~df_stream['anomaly']]['energy_score'].values
ax.boxplot([anom_es, normal_es], labels=['Anomali', 'Normal'], patch_artist=True, boxprops=dict(facecolor='#cfe2f3'), medianprops=dict(color='red'), whiskerprops=dict(linewidth=1.5))
ax.set_ylabel('Energy Score')
ax.set_title('Distribusi Energy Score: Anomali vs Normal', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# (3) Rolling R²
ax = fig.add_subplot(gs[2])
roll_w = min(1000, len(df_stream) // 2000)
if roll_w > 0:
    df_stream['r2_ma'] = df_stream['r2_running'].rolling(window=roll_w, min_periods=1).mean()
    ax.plot(df_stream['r2_ma'], linewidth=1.5, color='forestgreen')
ax.axhline(0.9, color='crimson', linestyle='--', linewidth=1.2, alpha=0.8)
ax.set_xlabel('Sample Index')
ax.set_ylabel('Rolled R²')
ax.set_title(f'Running R² — Moving Average (W={roll_w})', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('analysis_v6.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: analysis_v6.png')
